# Xây Dựng & Chuẩn Hóa Bảng Bản Ghi Nguồn: `job_source_records`

Notebook này tập trung xây dựng bảng quản lý bản ghi thu thập theo từng website nguồn **`job_source_records`** trong kiến trúc Silver Layer:
- Phân biệt rõ ràng giữa thực thể tin tuyển dụng chuẩn (`jobs`) và bản ghi vật lý được cào từ từng website tuyển dụng.
- Quản lý định danh bản ghi gốc (`source_job_id`), đường dẫn gốc (`source_job_url`), và thời điểm thu thập dữ liệu (`crawled_at`).
- Hỗ trợ kiến trúc Deduplication đa sàn: một vị trí tuyển dụng chuẩn có thể được liên kết với nhiều bản ghi nguồn khác nhau (TopCV, TopDev, VietnamWorks).
- Kiểm tra nghiêm ngặt tính toàn vẹn ràng buộc DDL PostgreSQL: khóa chính `source_record_id`, ràng buộc duy nhất `UNIQUE(source_id, source_job_id)`, và các khóa ngoại tham chiếu tới `jobs` và `job_sources`.

In [1]:
import os
import json
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Môi trường xử lý bảng job_source_records đã sẵn sàng!')

Môi trường xử lý bảng job_source_records đã sẵn sàng!


## 1. Nạp Dữ Liệu Bronze & Thiết Lập Ánh Xạ Khóa Ngoại

Đồng bộ thứ tự bản ghi với bảng `jobs` (`02_jobs.ipynb`) và lấy mã nguồn từ bảng `job_sources` (`06_job_sources.ipynb`).

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
if not os.path.exists(listings_path):
    listings_path = os.path.abspath('data/bronze/bronze_listings_master.json')

details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
if not os.path.exists(details_path):
    details_path = os.path.abspath('data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

df_listings = pd.DataFrame(bronze_listings)
df_details = pd.DataFrame(bronze_details)

df_merged = pd.merge(
    df_listings,
    df_details,
    on=['source', 'source_job_id'],
    how='inner',
    suffixes=('_list', '_detail')
)

df_merged['job_id'] = df_merged.index + 1
source_map = {'topcv': 1, 'topdev': 2, 'vietnamworks': 3, 'itviec': 4}
df_merged['source_id'] = df_merged['source'].map(source_map)

print(f'Tổng số bản ghi đồng bộ: {len(df_merged):,} bản ghi')
print(f'Toàn vẹn mã nguồn source_id: {df_merged["source_id"].notna().sum()} / {len(df_merged)} (100.0% hợp lệ)')


Tổng số bản ghi đồng bộ: 2,509 bản ghi
Toàn vẹn mã nguồn source_id: 2509 / 2509 (100.0% hợp lệ)


## 2. Kiểm Tra Chất Lượng Trường Dữ Liệu & Ràng Buộc Unique

In [3]:
pairs = list(zip(df_merged['source_id'], df_merged['source_job_id'].astype(str)))
unique_pairs = len(set(pairs))
is_unique_pair = unique_pairs == len(df_merged)

urls = df_merged['job_url_list'].fillna('').astype(str)
max_url_len = urls.apply(len).max()
avg_url_len = int(urls.apply(len).mean())
empty_urls = (urls == '').sum()

print('--- KIỂM TRA CHẤT LƯỢNG CÁC TRƯỜNG CỦA job_source_records ---')
print(f'Tính duy nhất của cặp (source_id, source_job_id): {is_unique_pair} ({unique_pairs:,} / {len(df_merged):,})')
print(f'Số lượng URL bị rỗng: {empty_urls}')
print(f'Độ dài URL tối đa: {max_url_len} ký tự (tuân thủ giới hạn VARCHAR(500))')
print(f'Độ dài URL trung bình: {avg_url_len} ký tự')

--- KIỂM TRA CHẤT LƯỢNG CÁC TRƯỜNG CỦA job_source_records ---
Tính duy nhất của cặp (source_id, source_job_id): True (2,509 / 2,509)
Số lượng URL bị rỗng: 0
Độ dài URL tối đa: 215 ký tự (tuân thủ giới hạn VARCHAR(500))
Độ dài URL trung bình: 85 ký tự


## 3. Khởi Tạo Bảng `job_source_records` Chuẩn DDL PostgreSQL

In [4]:
records = []
now_iso = datetime.now().isoformat()

for idx, row in df_merged.iterrows():
    source_record_id = idx + 1
    job_id = row['job_id']
    source_id = row['source_id']
    source_job_id = str(row['source_job_id']).strip()
    source_job_url = str(row['job_url_list'] or '').strip()
    crawled_at = row.get('crawled_at_list') or now_iso
    
    records.append({
        'source_record_id': source_record_id,
        'job_id': job_id,
        'source_id': source_id,
        'source_job_id': source_job_id,
        'source_job_url': source_job_url,
        'crawled_at': crawled_at,
        'created_at': now_iso,
        'updated_at': now_iso
    })

df_job_source_records = pd.DataFrame(records)

print('Khởi tạo bảng job_source_records thành công:')
print(f'  - Tổng số bản ghi: {len(df_job_source_records):,}')
print(f'  - Tính duy nhất của Primary Key source_record_id: {df_job_source_records["source_record_id"].is_unique}')
print(f'  - Toàn vẹn khóa ngoại job_id: {df_job_source_records["job_id"].between(1, len(df_merged)).all()} (100.0% hợp lệ)')
print(f'  - Toàn vẹn khóa ngoại source_id: {df_job_source_records["source_id"].isin([1, 2, 3]).all()} (100.0% hợp lệ)')
print(f'  - Tính duy nhất của Unique Constraint (source_id, source_job_id): {len(df_job_source_records) == len(df_job_source_records.drop_duplicates(subset=["source_id", "source_job_id"]))}')


Khởi tạo bảng job_source_records thành công:
  - Tổng số bản ghi: 2,509
  - Tính duy nhất của Primary Key source_record_id: True
  - Toàn vẹn khóa ngoại job_id: True (100.0% hợp lệ)
  - Toàn vẹn khóa ngoại source_id: True (100.0% hợp lệ)
  - Tính duy nhất của Unique Constraint (source_id, source_job_id): True


## 4. Phân Tích Thống Kê & Xem Trước Bảng `job_source_records`

In [5]:
print('--- 1. CẤU TRÚC BẢNG job_source_records (INFO) ---')
print(df_job_source_records.info())

print('\n--- 2. PHÂN BỐ BẢN GHI THEO NGUỒN THU THẬP (source_id) ---')
id_to_name = {1: 'TopCV', 2: 'TopDev', 3: 'VietnamWorks', 4: 'ITViec'}
df_source_dist = df_job_source_records['source_id'].map(id_to_name).value_counts()
df_source_summary = pd.DataFrame({
    'Số bản ghi': df_source_dist,
    'Tỷ lệ (%)': (df_source_dist / len(df_job_source_records) * 100).round(2)
})
print(df_source_summary.to_string())

print('\n--- 3. PREVIEW 10 DÒNG BẢNG job_source_records (CHUẨN POSTGRESQL) ---')
cols_preview = ['source_record_id', 'job_id', 'source_id', 'source_job_id', 'source_job_url', 'crawled_at']
print(df_job_source_records[cols_preview].head(10).to_string(index=False))


--- 1. CẤU TRÚC BẢNG job_source_records (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   source_record_id  2509 non-null   int64
 1   job_id            2509 non-null   int64
 2   source_id         2509 non-null   int64
 3   source_job_id     2509 non-null   str  
 4   source_job_url    2509 non-null   str  
 5   crawled_at        2509 non-null   str  
 6   created_at        2509 non-null   str  
 7   updated_at        2509 non-null   str  
dtypes: int64(3), str(5)
memory usage: 573.9 KB
None

--- 2. PHÂN BỐ BẢN GHI THEO NGUỒN THU THẬP (source_id) ---
              Số bản ghi  Tỷ lệ (%)
source_id                          
TopCV               1306      52.05
VietnamWorks        1079      43.01
TopDev               124       4.94

--- 3. PREVIEW 10 DÒNG BẢNG job_source_records (CHUẨN POSTGRESQL) ---
 source_record_id  job_id  source_id sou